# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [18]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\notebooks
프로젝트 루트: C:\Users\ojh23\Documents\projects\llm-data-analysis-course
데이터 폴더: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [19]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [20]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [21]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [22]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [23]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [24]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [25]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [26]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [27]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [28]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [29]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 13. 결과 저장하기


In [30]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 442
ch04_customer_sales.csv True 3387


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


### LLM Prompt

다음 온라인 쇼핑몰 데이터로 “어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?”를 분석하려고 합니다.

실제 컬럼은 다음과 같습니다.

- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price
- products: product_id, product_name, category, price

확인된 사실:
- order_status 값: completed, cancelled, refunded
- completed 주문만 분석 대상으로 사용한다.
- 금액 계산식: line_total = quantity × unit_price
- orders.order_id는 중복 0건이다.
- order_items와 orders의 병합은 order_id 기준 many_to_one 관계여야 한다.
- 카테고리별 주문 금액과 월별 주문 금액을 계산하고 싶다.
- 월별 주문 건수는 order_id의 고유 개수로 계산해야 한다.

위 조건을 만족하는 pandas 분석 코드의 작성 순서를 제안해 주세요.
실제 컬럼에 없는 이름을 가정하지 말고, merge 검증과 총합 검증도 포함해 주세요.

### LLM 제안 코드 요약

LLM은 다음과 같은 순서를 제안했다.

```python
completed_orders = orders[
    orders["order_status"] == "completed"
].copy()

sales = order_items.merge(
    completed_orders[["order_id", "customer_id", "order_date"]],
    on="order_id",
    how="inner",
)

sales = sales.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
)

sales["line_total"] = sales["quantity"] * sales["unit_price"]

category_sales = (
    sales.groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        total_quantity=("quantity", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

monthly_sales = (
    sales.groupby("order_date", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "count"),
    )
)

In [31]:
# LLM 제안 코드 검증

invalid_order_ids = order_items.loc[
    ~order_items["order_id"].isin(orders["order_id"]),
    "order_id",
].nunique()

invalid_product_ids = completed_order_sales.loc[
    ~completed_order_sales["product_id"].isin(products["product_id"]),
    "product_id",
].nunique()

monthly_order_check = (
    completed_order_sales
    .groupby("order_month")["order_id"]
    .nunique()
    .sort_index()
    .equals(
        monthly_summary
        .set_index("order_month")["order_count"]
        .sort_index()
    )
)

llm_code_validation = pd.DataFrame([
    {
        "검토 항목": "주문 상태 컬럼",
        "실제 데이터 확인": "order_status" in orders.columns,
        "판단": "사용",
    },
    {
        "검토 항목": "completed 필터 값",
        "실제 데이터 확인": sorted(orders["order_status"].unique()),
        "판단": "completed 값이 실제로 존재하므로 사용",
    },
    {
        "검토 항목": "orders.order_id 중복",
        "실제 데이터 확인": orders["order_id"].duplicated().sum(),
        "판단": "0건이므로 many_to_one 병합 가능",
    },
    {
        "검토 항목": "주문 병합 미매칭 ID",
        "실제 데이터 확인": invalid_order_ids,
        "판단": "0건이므로 현재 데이터에서는 정상 연결",
    },
    {
        "검토 항목": "상품 병합 미매칭 ID",
        "실제 데이터 확인": invalid_product_ids,
        "판단": "0건이므로 현재 데이터에서는 정상 연결",
    },
    {
        "검토 항목": "월별 주문 건수 기준",
        "실제 데이터 확인": monthly_order_check,
        "판단": "order_id.nunique() 기준으로 확인",
    },
    {
        "검토 항목": "총합 일치 검증",
        "실제 데이터 확인": completed_order_sales["line_total"].sum(),
        "판단": "15번 실습 과제에서 집계별 합계와 비교 예정",
    },
])

display(llm_code_validation)

,검토 항목,실제 데이터 확인,판단
0,주문 상태 컬럼,True,사용
1,completed 필터 값,"[cancelled, completed, refunded]",completed 값이 실제로 존재하므로 사용
2,orders.order_id 중복,0,0건이므로 many_to_one 병합 가능
3,주문 병합 미매칭 ID,0,0건이므로 현재 데이터에서는 정상 연결
4,상품 병합 미매칭 ID,0,0건이므로 현재 데이터에서는 정상 연결
5,월별 주문 건수 기준,True,order_id.nunique() 기준으로 확인
6,총합 일치 검증,148990000,15번 실습 과제에서 집계별 합계와 비교 예정


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [32]:
# 1. 40세 이상 고객 추출
customers_over_40 = customers[customers["age"] >= 40]

print("[과제 1] 40세 이상 고객")
display(customers_over_40.head())
print("40세 이상 고객 수:", len(customers_over_40))


# 2. 상품 가격이 낮은 순서대로 10개 출력
low_price_products = products.sort_values("price", ascending=True).head(10)

print("\n[과제 2] 가격이 낮은 상품 10개")
display(low_price_products)


# 3. 결제수단별 주문 수와 비율
payment_summary = (
    orders["payment_method"]
    .value_counts()
    .rename_axis("payment_method")
    .reset_index(name="order_count")
)

payment_summary["order_ratio_pct"] = (
    payment_summary["order_count"] / payment_summary["order_count"].sum() * 100
).round(2)

print("\n[과제 3] 결제수단별 주문 수와 비율")
display(payment_summary)


# 4. 카테고리별 평균 상품 가격
category_avg_price = (
    products
    .groupby("category", as_index=False)
    .agg(
        product_count=("product_id", "count"),
        average_price=("price", "mean"),
    )
    .sort_values("average_price", ascending=False)
)

category_avg_price["average_price"] = category_avg_price["average_price"].round(0)

print("\n[과제 4] 카테고리별 평균 상품 가격")
display(category_avg_price)


# 5. 완료 주문과 전체 주문의 금액 차이
completed_order_amount = completed_order_sales["line_total"].sum()

order_amount_comparison = pd.DataFrame([
    {"scope": "전체 주문 상세 금액", "amount": all_order_amount},
    {"scope": "completed 주문 금액", "amount": completed_order_amount},
    {"scope": "차이", "amount": all_order_amount - completed_order_amount},
])

print("\n[과제 5] completed 주문과 전체 주문 금액 비교")
display(order_amount_comparison)


# 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트
customer_sales_prompt = """
온라인 쇼핑몰 데이터에서 completed 주문 기준 고객별 구매 금액을 계산하는
pandas 코드를 작성해 주세요.

실제 컬럼:
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price
- customers: customer_id, name, gender, age, city, signup_date

분석 기준:
- order_status == "completed" 주문만 사용
- line_total = quantity * unit_price
- 고객별 주문 건수는 order_id.nunique()로 계산
- 고객별 total_sales는 line_total의 합계로 계산
- order_items와 orders 병합 시 order_id 기준으로
  validate="many_to_one", indicator=True를 사용
- 병합 전후 행 수와 미매칭 행을 확인
- 고객 이름은 결과에 포함하지 말고 customer_id와 city만 사용
"""

print("\n[과제 6] LLM 코드 요청 프롬프트")
print(customer_sales_prompt)


[과제 1] 40세 이상 고객


,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
6,7,이상현,F,53,인천,2024-12-12
8,9,송지민,M,69,서울,2025-10-19
9,10,유도현,F,62,울산,2024-10-06


40세 이상 고객 수: 78

[과제 2] 가격이 낮은 상품 10개


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000



[과제 3] 결제수단별 주문 수와 비율


...


[과제 4] 카테고리별 평균 상품 가격


,category,product_count,average_price
4,식품,7,137143.0
1,뷰티,16,117688.0
6,패션,11,115909.0
3,스포츠,19,111579.0
0,도서,14,106857.0
5,전자기기,17,101588.0
2,생활용품,16,96438.0



[과제 5] completed 주문과 전체 주문 금액 비교


,scope,amount
0,전체 주문 상세 금액,255610000
1,completed 주문 금액,148990000
2,차이,106620000



[과제 6] LLM 코드 요청 프롬프트

온라인 쇼핑몰 데이터에서 completed 주문 기준 고객별 구매 금액을 계산하는
pandas 코드를 작성해 주세요.

실제 컬럼:
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price
- customers: customer_id, name, gender, age, city, signup_date

분석 기준:
- order_status == "completed" 주문만 사용
- line_total = quantity * unit_price
- 고객별 주문 건수는 order_id.nunique()로 계산
- 고객별 total_sales는 line_total의 합계로 계산
- order_items와 orders 병합 시 order_id 기준으로
  validate="many_to_one", indicator=True를 사용
- 병합 전후 행 수와 미매칭 행을 확인
- 고객 이름은 결과에 포함하지 말고 customer_id와 city만 사용



## 추가 검증. completed 주문 집계 총합 확인

In [33]:
# 동일한 completed 주문 범위의 집계 총합이 모두 일치하는지 검증

completed_total = completed_order_sales["line_total"].sum()

total_check = pd.DataFrame([
    {
        "집계 기준": "원본 completed line_total",
        "total_sales": completed_total,
    },
    {
        "집계 기준": "카테고리별 total_sales 합계",
        "total_sales": category_sales["total_sales"].sum(),
    },
    {
        "집계 기준": "상품별 total_sales 합계",
        "total_sales": product_sales["total_sales"].sum(),
    },
    {
        "집계 기준": "월별 total_sales 합계",
        "total_sales": monthly_summary["total_sales"].sum(),
    },
    {
        "집계 기준": "고객별 total_sales 합계",
        "total_sales": customer_sales["total_sales"].sum(),
    },
])

total_check["원본과의 차이"] = total_check["total_sales"] - completed_total
total_check["일치 여부"] = total_check["원본과의 차이"].eq(0)

display(total_check)

assert total_check["일치 여부"].all(), "집계별 total_sales 합계가 일치하지 않습니다."
print("검증 완료: 모든 집계의 total_sales 합계가 일치합니다.")

,집계 기준,total_sales,원본과의 차이,일치 여부
0,원본 completed line_total,148990000,0,True
1,카테고리별 total_sales 합계,148990000,0,True
2,상품별 total_sales 합계,148990000,0,True
3,월별 total_sales 합계,148990000,0,True
4,고객별 total_sales 합계,148990000,0,True


검증 완료: 모든 집계의 total_sales 합계가 일치합니다.


## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 0. 제출 정보

- 이름: 오병희
- GitHub ID: OBHAR
- 작성일: 2026-09-25
- 최종 제출 URL: https://github.com/OBHAR/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

completed 주문 기준으로 가장 큰 주문 금액을 만든 상품 카테고리는 무엇인가?

### 사용한 파일/컬럼

- 핵심 분석 파일: `orders`, `order_items`, `products`
- 고객별 집계에 추가 사용한 파일: `customers`
- 주요 컬럼:
  - `orders`: `order_id`, `customer_id`, `order_date`, `order_status`
  - `order_items`: `order_id`, `product_id`, `quantity`, `unit_price`
  - `products`: `product_id`, `product_name`, `category`
- 포함 주문 상태: `order_status == "completed"`
- 금액 계산식: `line_total = quantity × unit_price`
- 최종 집계 기준: 카테고리별 `line_total` 합계(`total_sales`)

### 결과 관찰

completed 주문상세 474행을 기준으로 분석했으며, completed 주문 기준 금액은 148,990,000원이다. 카테고리별 집계 결과 스포츠 카테고리가 31,743,000원으로 가장 높았고, 전체 completed 주문 금액의 21.31%를 차지했다.

### 나의 해석과 판단

이번 질문의 핵심은 카테고리별 completed 주문 금액 비교이므로 `orders`, `order_items`, `products`가 필수다. `customers`는 질문 자체에는 필수는 아니지만 고객별 집계 결과를 만들기 위해 추가로 사용했다.

### 업무·분석적 의미

스포츠 카테고리는 현재 데이터 기준으로 completed 주문 금액이 가장 큰 카테고리이므로, 재고·상품 구성·후속 상세 분석의 우선 검토 후보가 될 수 있다.

### 한계와 추가 확인 사항

`total_sales`는 completed 주문의 수량×주문단가 합계일 뿐이며, 회계상 순매출·이익·상품 인기도를 의미하지 않는다. 할인, 쿠폰, 배송비, 세금, 환불 금액 처리 기준은 현재 데이터만으로 확인할 수 없다.

## 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: `order_status == "completed"`
- 정렬 기준: 상품 가격은 오름차순·내림차순으로 확인했고, 집계 결과는 `total_sales` 내림차순으로 정렬했다.
- 만든 파생 컬럼:
  - `line_total = quantity × unit_price`
  - `order_month = order_date`를 월 단위 문자열로 변환한 컬럼
- `line_total` 계산식: 주문상세 행별 수량과 주문 당시 단가를 곱해 계산했다.

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

전체 주문상세 764행 중 completed 주문상세는 474행이었다. 날짜 변환 실패는 0건이었으며, completed 주문의 `line_total` 합계는 148,990,000원이었다.

### 나의 해석과 판단

이번 분석에서는 completed 주문만 포함했으므로 cancelled·refunded 주문상세는 집계에서 제외됐다. 주문 상태 기준을 변경하면 포함되는 주문 행과 `total_sales`가 함께 달라진다.

### 업무·분석적 의미

주문상태와 금액 계산식을 먼저 명확히 하면 이후 카테고리·월·고객별 결과가 같은 분석 범위를 유지하도록 관리할 수 있다.

### 한계와 추가 확인 사항

`line_total`은 주문상세의 수량×단가 합계다. 할인, 쿠폰, 배송비, 세금, 환불 금액이 반영됐는지는 현재 데이터만으로 알 수 없다.

## 3. merge 검증

- 병합한 데이터:
  1. `order_items`와 `orders`
  2. completed 주문상세와 `products`
- 사용한 key:
  - 주문 병합: `order_id`
  - 상품 병합: `product_id`
- `validate` 결과: 두 병합 모두 `many_to_one` 관계로 정상 실행됐다.
- `indicator` 결과:
  - 주문 병합: `both` 764건, `left_only` 0건, `right_only` 0건
  - 상품 병합: `both` 474건, `left_only` 0건, `right_only` 0건
- 병합 전/후 행 수:
  - 주문 병합: 764행 → 764행
  - 상품 병합: 474행 → 474행

![merge 검증](images/step03_merge.png)

### 결과 관찰

`orders.order_id`와 `products.product_id`의 중복은 모두 0건이었다. 두 병합 모두 전후 행 수가 같고 미매칭 행도 없었다.

### 나의 해석과 판단

현재 데이터에서는 주문상세가 하나의 주문과 하나의 상품에 정상적으로 연결된다. 따라서 병합으로 인해 행이 불필요하게 늘어나거나, 카테고리가 누락되는 문제 없이 집계를 진행할 수 있다.

### 업무·분석적 의미

병합 검증 없이 집계하면 중복 행으로 금액이 과대 집계되거나 미매칭 행으로 금액이 누락될 수 있다. `validate`와 `indicator`는 이러한 문제를 집계 전에 찾는 안전장치다.

### 한계와 추가 확인 사항

현재는 키 존재 여부와 관계 형태를 확인한 것이다. 주문 취소·환불의 세부 처리 규칙, 상품 가격 변경 이력, 데이터 수집 시점 차이까지 검증한 것은 아니다.

## 4. completed 주문 범위와 집계

- 분석 범위 정의: `order_status == "completed"`인 주문의 주문상세만 사용했다.
- 카테고리별 결과: 스포츠가 31,743,000원으로 가장 높았으며, completed 주문 금액의 21.31%를 차지했다.
- 상품별 결과: 스포츠 상품 041이 5,705,000원으로 가장 높은 주문 금액을 기록했다.
- 월별 결과: 2025년 10월이 25,766,000원, 주문 26건으로 가장 높은 completed 주문 금액을 기록했다.
- 고객별 결과: Customer 74가 주문 5건, 총 4,110,000원으로 가장 높은 구매 금액을 기록했다.

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

카테고리별 `total_sales`는 스포츠, 전자기기, 생활용품 순으로 높았다. 월별 결과는 2025년 10월이 가장 높았고, 고객별 결과에서는 소수 고객의 구매 금액이 상대적으로 높게 나타났다.

### 나의 해석과 판단

스포츠는 completed 주문 기준 금액이 가장 큰 카테고리이므로 추가 분석 우선순위가 높다. 다만 주문 금액이 높다는 사실만으로 판매 수량이 가장 많거나 수익성이 가장 높다고 판단할 수는 없다.

### 업무·분석적 의미

카테고리·상품·월·고객별 결과는 재고 관리, 상품 구성 검토, 고객 세그먼트 분석의 출발점으로 활용할 수 있다. 이후에는 카테고리별 주문 수, 재구매율, 할인 여부 등을 함께 확인할 필요가 있다.

### 한계와 추가 확인 사항

2025년 7월과 2026년 7월은 전체 월이 아닐 수 있으므로 다른 완전한 월과 단순 비교하면 안 된다. 또한 `total_sales`에는 할인·쿠폰·배송비·세금·환불 회계 처리 정보가 포함되지 않는다.

## 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148,990,000원
- 카테고리 합계: 148,990,000원
- 상품별 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 고객별 합계: 148,990,000원
- 차이 여부: 모든 집계에서 차이 0원, 일치 여부 `True`

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

모든 집계가 동일한 completed 주문 범위와 `line_total`을 사용했음을 총합으로 확인했다. 따라서 현재 카테고리·상품·월·고객별 결과는 같은 분석 기준에서 비교할 수 있다.

합계가 일치하지 않았다면 먼저 필터 조건이 같은지, 병합으로 행이 증가했는지, 누락된 카테고리·상품·고객이 있는지, 집계 대상 컬럼이 동일한지 순서대로 확인해야 한다.

## 6. LLM pandas 코드 검증

- LLM Prompt 요약: completed 주문 기준 고객별 구매 금액을 계산하는 pandas 코드를 요청했다. 실제 컬럼명, `line_total` 계산식, `order_id.nunique()` 주문 건수 기준, `validate="many_to_one"`, `indicator=True` 사용을 조건으로 제시했다.
- 제안 코드 요약: completed 주문을 필터링하고 주문상세·주문·상품 데이터를 병합한 뒤 카테고리와 월별 금액을 집계하는 코드를 제안했다.
- 실제 컬럼/범위와 맞지 않거나 부족한 부분:
  - 병합 시 `validate`와 `indicator`가 없으면 관계와 미매칭 행을 확인할 수 없다.
  - 월별 주문 건수에 `order_id.count()`를 사용하면 주문 수가 아니라 주문상세 행 수가 된다.
  - 월별 집계 전에 날짜형 변환과 `order_month` 파생 컬럼 생성이 필요하다.
  - 집계별 total_sales 총합 일치 검증이 필요하다.
- 수정한 내용: 실제 `order_status` 컬럼과 `completed` 값을 사용하고, 두 병합에 `validate="many_to_one"`과 `indicator=True`를 적용했다. 주문 건수는 `order_id.nunique()`로 수정했고, 총합 검증을 추가했다.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

LLM 코드가 문법적으로 실행되더라도 분석 범위, 병합 관계, 주문 건수 기준이 맞지 않으면 결과가 왜곡될 수 있다. 따라서 실제 컬럼명과 출력 결과를 기준으로 수정한 뒤 사용해야 한다.

## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. 스포츠 카테고리가 completed 주문 기준 31,743,000원으로 가장 높았고, 전체 completed 주문 금액의 21.31%를 차지했다.
2. 원본 completed `line_total`과 카테고리·상품·월·고객별 집계 합계가 모두 148,990,000원으로 일치했다.

### 그 결과를 뒷받침하는 수치/표

- completed 주문상세: 474행
- completed 주문 기준 금액: 148,990,000원
- 스포츠 카테고리: 31,743,000원
- 2025년 10월: 25,766,000원, 주문 26건
- 총합 검증: 5개 집계 모두 차이 0원

### 추가로 확인하고 싶은 질문

- 스포츠 카테고리의 높은 주문 금액이 판매 수량, 높은 주문단가, 특정 상품 집중 중 무엇의 영향인지 확인하고 싶다.
- 고객별 구매 금액 상위 고객이 반복 구매 고객인지, 특정 기간의 일회성 고액 구매인지 확인하고 싶다.
- cancelled·refunded 주문을 포함했을 때 카테고리별 금액 순위가 어떻게 달라지는지 확인하고 싶다.

### 현재 결과의 한계

현재 `total_sales`는 completed 주문상세의 수량×주문단가 합계다. 순매출, 이익, 할인·쿠폰·배송비·세금·환불 반영 금액으로 해석할 수 없다. 또한 2025년 7월과 2026년 7월은 전체 월이 아닐 수 있으므로 월별 비교 시 주의해야 한다.

## 최종 제출 체크

- [x] 핵심 셀 Output이 남아 있습니다.
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.